# 項目1: テーマ・エクスポージャの構築

この notebook は Phase B の項目1だけを検証するための切り出しである。目的は、point-in-time のテーマバスケット構成から、銘柄別テーマ構成行列 `A_t` とテーマ・エクスポージャ行列 `X_M,t` を作ることに限定する。

ここでは Barra 既知因子からの純化 `Z = X_{M\perp V}`、adjusted R²、テーマリターン推定、コヒーレンス判定、最適化は扱わない。それらは項目2以降で確認する。


## 必要データ形式

この notebook は手元の実データのみで実行する。合成データは含めない。

| 変数 | 形式 | 必須内容 |
|---|---|---|
| `stock_returns` | wide | index=取引日、columns=銘柄ID、values=日次トータルリターン |
| `theme_basket_weights` | long | `effective_date`, `theme_id`, `security_id`, `weight`。`available_date` は任意 |
| `barra_exposures` | long | `date`, `security_id`, `factor_id`, `exposure` |
| `barra_regression_weights` | long | `date`, `security_id`, `regression_weight` |
| `barra_estimation_universe` | long | `date`, `security_id`, `in_estimation_universe` |

`available_date` がない場合は `effective_date` を情報利用可能日として扱う。各テーマ構成は `knowledge_date = max(effective_date, available_date)` で point-in-time 管理する。

期待する主な出力は次の4つである。

| 出力 | 内容 |
|---|---|
| `A_t` | 実装可能なテーマバスケット構成行列。列合計は1 |
| `coverage` | 推定対象ユニバースに残った各テーマ構成ウェイトの比率 |
| `X_M_t` | `theme_exposure_mode` に従って作成し、回帰ウェイトで単位ノルム化したテーマ・エクスポージャ |
| `diagnostics` | 日付、銘柄フィルタ、テーマフィルタ、列合計、ノルムの確認表 |


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Literal, Mapping, Optional, Sequence, Tuple
import math

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


@dataclass
class ThemeExposureConfig:
    asof_date: str = "YYYY-MM-DD"
    exposure_lag_days: int = 1
    theme_exposure_mode: Literal["basket_weight", "membership", "rank_weight"] = "basket_weight"
    min_basket_coverage: float = 0.80

    def validate(self) -> None:
        if self.asof_date == "YYYY-MM-DD":
            raise ValueError("cfg.asof_date を実データで確認したい基準日に変更してください。")
        if self.exposure_lag_days < 0:
            raise ValueError("exposure_lag_days must be nonnegative.")
        if self.theme_exposure_mode not in {"basket_weight", "membership", "rank_weight"}:
            raise ValueError("theme_exposure_mode must be basket_weight, membership, or rank_weight.")
        if not 0 <= self.min_basket_coverage <= 1:
            raise ValueError("min_basket_coverage must lie in [0, 1].")


cfg = ThemeExposureConfig(
    asof_date="YYYY-MM-DD",
    exposure_lag_days=1,
    theme_exposure_mode="basket_weight",
    min_basket_coverage=0.80,
)

# "files" の場合は下の paths を編集する。
# "in_memory" の場合は、この notebook 実行前に同名DataFrameをメモリへ用意する。
DATA_MODE: Literal["files", "in_memory"] = "files"

paths = {
    "stock_returns": "data/stock_returns.csv",
    "theme_basket_weights": "data/theme_basket_weights.csv",
    "barra_exposures": "data/barra_exposures.csv",
    "barra_regression_weights": "data/barra_regression_weights.csv",
    "barra_estimation_universe": "data/barra_estimation_universe.csv",
}


## データ読込・正規化

wide型CSVは、`date` 列があればそれをindexにする。`date` 列がない場合は先頭列を日付列として扱う。Parquet、Feather、pickle も読めるようにしているが、追加ライブラリがない環境ではCSVを使う。


In [ ]:
def _read_table(path: str | Path, wide: bool = False) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt"}:
        df = pd.read_csv(path)
    elif suffix in {".pkl", ".pickle"}:
        df = pd.read_pickle(path)
    elif suffix in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    elif suffix == ".feather":
        df = pd.read_feather(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    if wide:
        date_col = "date" if "date" in df.columns else df.columns[0]
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)
    return df


def _normalise_wide(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index)).tz_localize(None)
    out = out[~out.index.duplicated(keep="last")].sort_index()
    out.columns = out.columns.map(str)
    out = out.apply(pd.to_numeric, errors="coerce")
    if out.empty:
        raise ValueError(f"{name} is empty.")
    return out.astype(float)


def _normalise_long_dates(df: pd.DataFrame, date_cols: Sequence[str]) -> pd.DataFrame:
    out = df.copy()
    for col in date_cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col]).dt.tz_localize(None)
    return out


def _coerce_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0.0) != 0.0
    return series.astype(str).str.strip().str.lower().isin({"true", "t", "1", "yes", "y"})


def load_raw_inputs(paths: Mapping[str, str], data_mode: str) -> Dict[str, pd.DataFrame]:
    names = [
        "stock_returns",
        "theme_basket_weights",
        "barra_exposures",
        "barra_regression_weights",
        "barra_estimation_universe",
    ]
    if data_mode == "files":
        return {
            "stock_returns": _read_table(paths["stock_returns"], wide=True),
            "theme_basket_weights": _read_table(paths["theme_basket_weights"]),
            "barra_exposures": _read_table(paths["barra_exposures"]),
            "barra_regression_weights": _read_table(paths["barra_regression_weights"]),
            "barra_estimation_universe": _read_table(paths["barra_estimation_universe"]),
        }
    if data_mode == "in_memory":
        missing = [name for name in names if name not in globals()]
        if missing:
            raise NameError(f"DATA_MODE='in_memory' ですが、未定義のDataFrameがあります: {missing}")
        return {name: globals()[name].copy() for name in names}
    raise ValueError("DATA_MODE must be 'files' or 'in_memory'.")


def validate_and_normalise_inputs(raw: Mapping[str, pd.DataFrame], cfg: ThemeExposureConfig) -> Dict[str, pd.DataFrame]:
    cfg.validate()

    stock_returns = _normalise_wide(raw["stock_returns"], "stock_returns")

    theme_basket_weights = _normalise_long_dates(raw["theme_basket_weights"], ["effective_date", "available_date"])
    required_bw = {"effective_date", "theme_id", "security_id", "weight"}
    missing = required_bw - set(theme_basket_weights.columns)
    if missing:
        raise ValueError(f"theme_basket_weights is missing columns: {sorted(missing)}")
    if "available_date" not in theme_basket_weights.columns:
        theme_basket_weights["available_date"] = theme_basket_weights["effective_date"]
    else:
        theme_basket_weights["available_date"] = theme_basket_weights["available_date"].fillna(
            theme_basket_weights["effective_date"]
        )
    theme_basket_weights["knowledge_date"] = theme_basket_weights[["effective_date", "available_date"]].max(axis=1)
    theme_basket_weights["theme_id"] = theme_basket_weights["theme_id"].astype(str)
    theme_basket_weights["security_id"] = theme_basket_weights["security_id"].astype(str)
    theme_basket_weights["weight"] = pd.to_numeric(theme_basket_weights["weight"], errors="coerce")
    if theme_basket_weights["weight"].isna().any():
        raise ValueError("theme_basket_weights.weight contains nonnumeric values.")
    if (theme_basket_weights["weight"] < -1e-12).any():
        raise ValueError("Theme basket weights must be nonnegative.")
    theme_basket_weights = theme_basket_weights[theme_basket_weights["weight"] > 0].copy()
    sums = theme_basket_weights.groupby(["knowledge_date", "theme_id"])["weight"].transform("sum")
    if (sums <= 0).any():
        raise ValueError("At least one theme snapshot has nonpositive total weight.")
    theme_basket_weights["weight"] = theme_basket_weights["weight"] / sums

    barra_exposures = _normalise_long_dates(raw["barra_exposures"], ["date"])
    required_ex = {"date", "security_id", "factor_id", "exposure"}
    missing = required_ex - set(barra_exposures.columns)
    if missing:
        raise ValueError(f"barra_exposures is missing columns: {sorted(missing)}")
    barra_exposures["security_id"] = barra_exposures["security_id"].astype(str)
    barra_exposures["factor_id"] = barra_exposures["factor_id"].astype(str)
    barra_exposures["exposure"] = pd.to_numeric(barra_exposures["exposure"], errors="coerce")

    barra_regression_weights = _normalise_long_dates(raw["barra_regression_weights"], ["date"])
    required_rw = {"date", "security_id", "regression_weight"}
    missing = required_rw - set(barra_regression_weights.columns)
    if missing:
        raise ValueError(f"barra_regression_weights is missing columns: {sorted(missing)}")
    barra_regression_weights["security_id"] = barra_regression_weights["security_id"].astype(str)
    barra_regression_weights["regression_weight"] = pd.to_numeric(
        barra_regression_weights["regression_weight"], errors="coerce"
    )
    if (barra_regression_weights["regression_weight"].dropna() <= 0).any():
        raise ValueError("Barra regression weights must be positive.")

    barra_estimation_universe = _normalise_long_dates(raw["barra_estimation_universe"], ["date"])
    required_eu = {"date", "security_id", "in_estimation_universe"}
    missing = required_eu - set(barra_estimation_universe.columns)
    if missing:
        raise ValueError(f"barra_estimation_universe is missing columns: {sorted(missing)}")
    barra_estimation_universe["security_id"] = barra_estimation_universe["security_id"].astype(str)
    barra_estimation_universe["in_estimation_universe"] = _coerce_bool(
        barra_estimation_universe["in_estimation_universe"]
    )

    return {
        "stock_returns": stock_returns,
        "theme_basket_weights": theme_basket_weights,
        "barra_exposures": barra_exposures,
        "barra_regression_weights": barra_regression_weights,
        "barra_estimation_universe": barra_estimation_universe,
    }


## Point-in-time スナップショット

`SnapshotStore` は、指定日以前の直近スナップショットを返す。`BasketStore` はテーマごとに `knowledge_date <= asof_date` の最新構成だけを使う。


In [ ]:
class SnapshotStore:
    def __init__(self, by_date: Mapping[pd.Timestamp, Any]):
        self.by_date = {pd.Timestamp(k): v for k, v in by_date.items()}
        self.dates = pd.DatetimeIndex(sorted(self.by_date))
        if len(self.dates) == 0:
            raise ValueError("SnapshotStore requires at least one snapshot.")

    def get(self, asof: pd.Timestamp) -> Any:
        asof = pd.Timestamp(asof)
        pos = self.dates.searchsorted(asof, side="right") - 1
        if pos < 0:
            raise KeyError(f"No snapshot available on or before {asof}.")
        return self.by_date[self.dates[pos]]

    def get_with_date(self, asof: pd.Timestamp) -> Tuple[pd.Timestamp, Any]:
        asof = pd.Timestamp(asof)
        pos = self.dates.searchsorted(asof, side="right") - 1
        if pos < 0:
            raise KeyError(f"No snapshot available on or before {asof}.")
        d = self.dates[pos]
        return d, self.by_date[d]


def build_exposure_store(exposures: pd.DataFrame) -> SnapshotStore:
    by = {}
    for d, g in exposures.groupby("date", sort=True):
        x = g.pivot_table(index="security_id", columns="factor_id", values="exposure", aggfunc="last")
        x.index = x.index.astype(str)
        x.columns = x.columns.astype(str)
        by[pd.Timestamp(d)] = x.sort_index()
    return SnapshotStore(by)


def build_weight_store(weights: pd.DataFrame) -> SnapshotStore:
    by = {}
    for d, g in weights.groupby("date", sort=True):
        s = g.drop_duplicates("security_id", keep="last").set_index("security_id")["regression_weight"]
        s.index = s.index.astype(str)
        by[pd.Timestamp(d)] = s.astype(float)
    return SnapshotStore(by)


def build_universe_store(universe: pd.DataFrame) -> SnapshotStore:
    by = {}
    for d, g in universe.groupby("date", sort=True):
        s = g.drop_duplicates("security_id", keep="last").set_index("security_id")["in_estimation_universe"]
        s.index = s.index.astype(str)
        by[pd.Timestamp(d)] = s.astype(bool)
    return SnapshotStore(by)


class BasketStore:
    def __init__(self, basket_weights: pd.DataFrame):
        self.versions: Dict[str, list[Tuple[pd.Timestamp, pd.Series]]] = {}
        grouped = basket_weights.groupby(["theme_id", "knowledge_date"], sort=True)
        for (theme, d), g in grouped:
            s = g.groupby("security_id")["weight"].sum()
            s = s[s > 0]
            if s.sum() <= 0:
                continue
            s = s / s.sum()
            self.versions.setdefault(str(theme), []).append((pd.Timestamp(d), s.astype(float)))
        for theme in self.versions:
            self.versions[theme].sort(key=lambda x: x[0])
        self.themes = sorted(self.versions)

    def snapshot_with_dates(self, asof: pd.Timestamp) -> Tuple[pd.DataFrame, pd.Series]:
        asof = pd.Timestamp(asof)
        cols = {}
        dates = {}
        for theme, versions in self.versions.items():
            version_dates = pd.DatetimeIndex([v[0] for v in versions])
            pos = version_dates.searchsorted(asof, side="right") - 1
            if pos >= 0:
                cols[theme] = versions[pos][1]
                dates[theme] = versions[pos][0]
        if not cols:
            return pd.DataFrame(dtype=float), pd.Series(dtype="datetime64[ns]", name="basket_knowledge_date")
        a = pd.DataFrame(cols).fillna(0.0)
        sums = a.sum(axis=0)
        a = a.loc[:, sums > 0]
        a = a.div(a.sum(axis=0), axis=1).sort_index()
        date_series = pd.Series(dates, name="basket_knowledge_date").reindex(a.columns)
        return a, pd.to_datetime(date_series)

    def snapshot(self, asof: pd.Timestamp) -> pd.DataFrame:
        return self.snapshot_with_dates(asof)[0]


@dataclass
class Stores:
    exposures: SnapshotStore
    regression_weights: SnapshotStore
    estimation_universe: SnapshotStore
    baskets: BasketStore


def build_stores(data: Mapping[str, pd.DataFrame]) -> Stores:
    return Stores(
        exposures=build_exposure_store(data["barra_exposures"]),
        regression_weights=build_weight_store(data["barra_regression_weights"]),
        estimation_universe=build_universe_store(data["barra_estimation_universe"]),
        baskets=BasketStore(data["theme_basket_weights"]),
    )


## テーマ・エクスポージャ作成ロジック

元 notebook の `build_raw_theme_exposure` と同じく、`basket_weight`、`membership`、`rank_weight` を選べる。列ごとに回帰ウェイト $W$ による単位ノルム化を行う。


In [ ]:
def previous_trading_date(index: pd.DatetimeIndex, date: pd.Timestamp, lag: int = 1) -> pd.Timestamp:
    pos = index.searchsorted(pd.Timestamp(date), side="right") - 1 - lag
    if pos < 0:
        raise KeyError(f"Not enough history before {date} for lag={lag}.")
    return pd.Timestamp(index[pos])


def build_raw_theme_exposure(
    basket_snapshot: pd.DataFrame,
    universe: Sequence[str],
    mode: str,
    regression_weights: pd.Series,
) -> pd.DataFrame:
    a = basket_snapshot.reindex(index=list(universe), fill_value=0.0).astype(float)
    if mode == "membership":
        x = (a > 0).astype(float)
    elif mode == "rank_weight":
        x = a.rank(axis=0, pct=True, method="average").where(a > 0, 0.0)
    elif mode == "basket_weight":
        x = a.copy()
    else:
        raise ValueError(f"Unknown theme_exposure_mode={mode}")

    w = regression_weights.reindex(x.index).fillna(0.0).to_numpy(dtype=float)
    for col in x.columns:
        v = x[col].to_numpy(dtype=float)
        valid = np.isfinite(v) & (w > 0)
        if valid.sum() == 0:
            x[col] = 0.0
            continue
        # 中心化はしない。元notebookと同じく、市場/切片因子による処理は項目2のBarra純化へ送る。
        norm = math.sqrt(max(float(np.sum(w[valid] * v[valid] ** 2)), 0.0))
        if norm > 0:
            x[col] = v / norm
    return x


def weighted_column_norms(frame: pd.DataFrame, regression_weights: pd.Series) -> pd.Series:
    w = regression_weights.reindex(frame.index).fillna(0.0).to_numpy(dtype=float)
    out = {}
    for col in frame.columns:
        v = frame[col].to_numpy(dtype=float)
        valid = np.isfinite(v) & np.isfinite(w) & (w > 0)
        out[col] = math.sqrt(max(float(np.sum(w[valid] * v[valid] ** 2)), 0.0)) if valid.any() else np.nan
    return pd.Series(out, dtype=float, name="weighted_norm")


def construct_theme_exposure_snapshot(
    data: Mapping[str, pd.DataFrame],
    stores: Stores,
    cfg: ThemeExposureConfig,
) -> Dict[str, Any]:
    cfg.validate()
    asof = pd.Timestamp(cfg.asof_date)
    trading_index = data["stock_returns"].index
    exp_date = previous_trading_date(trading_index, asof, cfg.exposure_lag_days)

    X_full = stores.exposures.get(exp_date)
    rw = stores.regression_weights.get(exp_date)
    eu = stores.estimation_universe.get(exp_date)
    A_raw, basket_dates = stores.baskets.snapshot_with_dates(asof)
    if A_raw.empty:
        raise ValueError(f"No theme baskets available on or before {asof}.")

    return_universe = pd.Index(data["stock_returns"].columns)
    base_secs = return_universe.intersection(X_full.index)

    eu_aligned = eu.reindex(base_secs).fillna(False)
    secs_after_eu = base_secs[eu_aligned.to_numpy(dtype=bool)]

    rw_aligned_step = rw.reindex(secs_after_eu)
    valid_w = rw_aligned_step.notna() & (rw_aligned_step > 0)
    secs_after_weight = secs_after_eu[valid_w.to_numpy()]

    X = X_full.reindex(secs_after_weight)
    complete_x = X.notna().all(axis=1)
    eligible_secs = secs_after_weight[complete_x.to_numpy()]
    X = X.loc[eligible_secs]
    rw_aligned = rw.reindex(eligible_secs).astype(float)

    original_sum = A_raw.sum(axis=0).replace(0.0, np.nan)
    A_before_filter = A_raw.reindex(index=eligible_secs, fill_value=0.0)
    coverage_all = (A_before_filter.sum(axis=0) / original_sum).replace([np.inf, -np.inf], np.nan)

    keep_themes = coverage_all[coverage_all >= cfg.min_basket_coverage].index
    A_t = A_before_filter.loc[:, keep_themes]
    A_t = A_t.loc[:, A_t.sum(axis=0) > 0]
    if A_t.empty:
        raise ValueError("No theme passes min_basket_coverage after universe/exposure filters.")
    A_t = A_t.div(A_t.sum(axis=0), axis=1)
    coverage = coverage_all.reindex(A_t.columns)

    X_M_t = build_raw_theme_exposure(A_t, eligible_secs, cfg.theme_exposure_mode, rw_aligned)

    filter_counts = pd.Series(
        {
            "stock_return_columns": len(return_universe),
            "with_barra_exposure_snapshot": len(base_secs),
            "in_estimation_universe": len(secs_after_eu),
            "positive_regression_weight": len(secs_after_weight),
            "complete_barra_exposure": len(eligible_secs),
        },
        name="n_securities",
    )

    theme_filter = pd.DataFrame(
        {
            "raw_weight_sum_before_filter": original_sum,
            "coverage_after_security_filters": coverage_all,
            "kept": pd.Series(A_raw.columns.isin(A_t.columns), index=A_raw.columns),
            "basket_knowledge_date": basket_dates.reindex(A_raw.columns),
        }
    )
    theme_filter.index.name = "theme_id"

    date_diagnostics = pd.Series(
        {
            "asof_date": asof,
            "exposure_date_used": exp_date,
            "latest_basket_knowledge_date_used": basket_dates.reindex(A_t.columns).max(),
            "raw_theme_count": A_raw.shape[1],
            "kept_theme_count": A_t.shape[1],
            "raw_basket_security_count": A_raw.shape[0],
            "eligible_security_count": len(eligible_secs),
        },
        name="value",
    )

    diagnostics = {
        "date": date_diagnostics,
        "security_filter_counts": filter_counts,
        "theme_filter": theme_filter,
        "A_column_sum": A_t.sum(axis=0).rename("A_column_sum"),
        "X_M_weighted_norm": weighted_column_norms(X_M_t, rw_aligned),
    }

    return {
        "asof_date": asof,
        "exposure_date": exp_date,
        "A_raw": A_raw,
        "A_t": A_t,
        "coverage": coverage.rename("coverage"),
        "X_M_t": X_M_t,
        "regression_weights": rw_aligned,
        "eligible_securities": pd.Index(eligible_secs),
        "diagnostics": diagnostics,
    }


## 実行セル

`cfg.asof_date` と `paths` またはメモリ上のDataFrameを設定してから実行する。


In [ ]:
raw_inputs = load_raw_inputs(paths, DATA_MODE)
data = validate_and_normalise_inputs(raw_inputs, cfg)
stores = build_stores(data)

snapshot = construct_theme_exposure_snapshot(data, stores, cfg)

A_t = snapshot["A_t"]
coverage = snapshot["coverage"]
X_M_t = snapshot["X_M_t"]
diagnostics = snapshot["diagnostics"]

print(f"asof_date: {snapshot['asof_date'].date()}")
print(f"exposure_date_used: {snapshot['exposure_date'].date()}")
print(f"A_t shape: {A_t.shape}")
print(f"X_M_t shape: {X_M_t.shape}")


## 出力確認

`A_t` は実際に売買対象となるテーマバスケット構成行列、`X_M_t` は項目1で構築するテーマ・エクスポージャ行列である。ここでは両者は同じ列集合を持つ。


In [ ]:
display(diagnostics["date"].to_frame())
display(diagnostics["security_filter_counts"].to_frame())
display(diagnostics["theme_filter"].sort_values(["kept", "coverage_after_security_filters"], ascending=[False, False]))

display(pd.concat([diagnostics["A_column_sum"], coverage, diagnostics["X_M_weighted_norm"]], axis=1))

display(A_t.head())
display(X_M_t.head())


## 受入チェック

次のチェックが通れば、項目1としては期待どおりである。

- `A_t` の各テーマ列合計が1
- 残ったテーマは `coverage >= min_basket_coverage`
- `X_M_t` の各テーマ列が回帰ウェイトで単位ノルム
- テーマ構成の `knowledge_date` が `asof_date` を超えない
- Barraエクスポージャ使用日は `asof_date` 以前


In [ ]:
tol = 1e-8

assert not A_t.empty, "A_t is empty."
assert not X_M_t.empty, "X_M_t is empty."
assert A_t.index.equals(X_M_t.index), "A_t and X_M_t must have the same security index."
assert A_t.columns.equals(X_M_t.columns), "A_t and X_M_t must have the same theme columns."
assert np.allclose(A_t.sum(axis=0).to_numpy(dtype=float), 1.0, atol=tol), "A_t columns must sum to 1."
assert (coverage >= cfg.min_basket_coverage - tol).all(), "All kept themes must pass min_basket_coverage."
assert np.allclose(
    diagnostics["X_M_weighted_norm"].dropna().to_numpy(dtype=float),
    1.0,
    atol=1e-8,
), "X_M_t columns must have weighted unit norm."
assert diagnostics["date"].loc["exposure_date_used"] <= diagnostics["date"].loc["asof_date"]
kept_theme_dates = diagnostics["theme_filter"].loc[A_t.columns, "basket_knowledge_date"]
assert (kept_theme_dates <= diagnostics["date"].loc["asof_date"]).all(), "Future basket information was used."

print("Item 1 acceptance checks passed.")


## 次の項目へ進む前の確認メモ

項目2では、この notebook で作った `X_M_t` を Barra既知因子 `X_V,t` へ回帰し、純化テーマ・エクスポージャ `Z_t = X_{M\perp V,t}` を作る。したがって、この段階では `X_M_t` を中心化しない。元 notebook と同じく、平均・市場・業種・スタイルで説明できる部分は項目2の純化で扱う。
